In [ ]:
# # RSAP Tools Manual Testing Notebook

# This notebook allows you to manually test all the tools available for controlling the RosSequentialActionProgrammer.

# ## Setup
# First, initialize the ROS2 node and create the Tools instance.

================================== Ai Message ==================================

Your name is Bob.


In [1]:
import rclpy
from rclpy.node import Node
import json
import sys

# Initialize ROS2
rclpy.init()

# Create a test node
test_node = Node('tools_test_node')
print("✓ ROS2 node created successfully")

✓ ROS2 node created successfully


In [2]:
# Import the Tools class
import sys
from pathlib import Path

# Add the package directory to sys.path
package_path = Path.cwd()
if str(package_path) not in sys.path:
	sys.path.insert(0, str(package_path))

from submodules.langchain.tools.Tools import Tools

# Create Tools instance
tools = Tools(test_node)
print("✓ Tools instance created successfully")
print(f"✓ RSAP instance ready with {len(tools.rsap.action_list)} actions in sequence")

✓ Tools instance created successfully
✓ RSAP instance ready with 0 actions in sequence


[INFO] [1770976615.966748029] [tools_test_node]: Config loaded from file: /home/match-mover/Documents/ros2_ws/install/ros_sequential_action_programmer/share/ros_sequential_action_programmer/rsap_config.yaml


---
## 1. Query Tools
Test tools that retrieve information about available services and current sequence.

In [24]:
# Test 1.1: Get Available Services
print("=" * 60)
print("TEST 1.1: Get Available Services")
print("=" * 60)

result = tools._get_available_services("")
data = json.loads(result)
print(f"Found {data.get('count', 0)} services:")
for service in data.get('services', [])[:5]:  # Show first 5
    print(f"  - {service['client']}: {service['type']}")
if data.get('count', 0) > 5:
    print(f"  ... and {data.get('count', 0) - 5} more")
print()

TEST 1.1: Get Available Services
Found 0 services:



In [5]:
# Test 1.2: Get Available ROS Actions
print("=" * 60)
print("TEST 1.2: Get Available ROS Actions")
print("=" * 60)

result = tools._get_available_ros_actions("")
data = json.loads(result)
print(f"Found {data.get('count', 0)} ROS actions:")
for action in data.get('actions', [])[:5]:  # Show first 5
    print(f"  - {action['client']}: {action['type']}")
if data.get('count', 0) > 5:
    print(f"  ... and {data.get('count', 0) - 5} more")
print()

TEST 1.2: Get Available ROS Actions
Found 7 ROS actions:
  - /execute_trajectory: moveit_msgs/action/ExecuteTrajectory
  - /move_action: moveit_msgs/action/MoveGroup
  - /pm_robot_gonio_left_controller/follow_joint_trajectory: control_msgs/action/FollowJointTrajectory
  - /pm_robot_gonio_right_controller/follow_joint_trajectory: control_msgs/action/FollowJointTrajectory
  - /pm_robot_t_axis_controller/follow_joint_trajectory: control_msgs/action/FollowJointTrajectory
  ... and 2 more



In [18]:
# Test 1.3: Get Current Action List
print("=" * 60)
print("TEST 1.3: Get Current Action List")
print("=" * 60)

result = tools._get_action_list("")
data = json.loads(result)
print(f"Current sequence has {data['total_count']} actions:")
for action in data['actions']:
    print(f"  [{action['index']}] {action['name']} ({action['type']}) - Active: {action['is_active']}")
if data['total_count'] == 0:
    print("  (Sequence is empty)")
print()

TEST 1.3: Get Current Action List
Current sequence has 0 actions:
  (Sequence is empty)



In [3]:
# Test 1.4: Get Service Parameters
print("=" * 60)
print("TEST 1.4: Get Service Parameters")
print("=" * 60)

# Test getting parameters for a specific service
service_client = "/pm_robot_primitive_skills/uv_curing"
print(f"Getting parameters for: {service_client}\n")


# Try to get parameters
result = tools._get_service_parameters(service_client)
data = json.loads(result)

if data.get('success'):
    print(f"✓ Found parameters for {data['count']} service(s)")
    for service_name, service_info in data.get('services', {}).items():
        print(f"\n  Service: {service_name}")
        print(f"  Type: {service_info.get('service_type', 'Unknown')}")
        
        # Request parameters
        request = service_info.get('request', {})
        if request:
            print(f"  Request parameters:")
            for param_name, param_type in request.items():
                print(f"    - {param_name}: {param_type}")
        else:
            print(f"  Request parameters: (none)")
        
        # Response parameters
        response = service_info.get('response', {})
        if response:
            print(f"  Response parameters:")
            for param_name, param_type in response.items():
                print(f"    - {param_name}: {param_type}")
        else:
            print(f"  Response parameters: (none)")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")
    if data.get('missing'):
        print(f"  Missing services: {data.get('missing')}")
    if data.get('requested'):
        print(f"  Requested services: {data.get('requested')}")
print()

TEST 1.4: Get Service Parameters
Getting parameters for: /pm_robot_primitive_skills/uv_curing

✗ Error: Service(s) not found in available services: /pm_robot_primitive_skills/uv_curing. Make sure the service(s) are running.
  Missing services: ['/pm_robot_primitive_skills/uv_curing']
  Requested services: ['/pm_robot_primitive_skills/uv_curing']



---
## 2. Building Sequence Tools
Test tools for adding actions to the sequence.

In [ ]:
# Test 2.1: Add Service to Sequence
print("=" * 60)
print("TEST 2.1: Add Service to Sequence")
print("=" * 60)

# First, get available services
services_result = tools._get_available_services("")
services_data = json.loads(services_result)

if services_data.get('count', 0) > 0:
    # Add the first 5 available services (or fewer if less than 5 exist)
    services_to_add = min(5, services_data['count'])
    
    print(f"Adding {services_to_add} services to the sequence...\n")
    
    for i in range(services_to_add):
        service = services_data['services'][i]
        service_client = service['client']
        service_type = service['type']
        
        input_json = json.dumps({
            "service_client": service_client,
            "index": i + 1,  # 1-based index
            "service_type": service_type,
            "service_name": f"Service {i+1}: {service_client.split('/')[-1]}"
        })
        
        result = tools._add_service_to_sequence(input_json)
        data = json.loads(result)
        
        if data.get('success'):
            print(f"✓ Added [{i+1}] {service_client}")
        else:
            print(f"✗ Failed to add {service_client}: {data.get('error', 'Unknown error')}")
    
    # Show final sequence length
    final_result = tools._get_sequence_summary("")
    final_data = json.loads(final_result)
    print(f"\n✓ Sequence now has {final_data['total_count']} actions")
else:
    print("✗ No services available to test")
print()

TEST 2.1: Add Service to Sequence
✓ Service '/gonio_orientation_solver/get_gonio_left_solution' added at index 0
  Sequence length: 1



[INFO] [1770652251.793123291] [tools_test_node]: Inserted Service for /gonio_orientation_solver/get_gonio_left_solution at 0


In [ ]:
# Test 2.2: Add User Interaction to Sequence
print("=" * 60)
print("TEST 2.2: Add User Interaction to Sequence")
print("=" * 60)

input_json = json.dumps({
    "index": 1,
    "action_name": "Confirm Position",
    "action_description": "Please confirm the robot is in the correct position before continuing",
    "interaction_mode": "terminal"
})

result = tools._add_user_interaction(input_json)
data = json.loads(result)

if data.get('success'):
    print(f"✓ {data['message']}")
    print(f"  Sequence length: {data['current_sequence_length']}")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")
print()

In [8]:
# Test 2.3: View Updated Sequence
print("=" * 60)
print("TEST 2.3: View Updated Sequence")
print("=" * 60)

result = tools._get_action_list("")
data = json.loads(result)
print(f"Sequence now has {data['total_count']} actions:")
for action in data['actions']:
    print(f"  [{action['index']}] {action['name']} ({action['type']})")
print()

TEST 2.3: View Updated Sequence
Sequence now has 1 actions:
  [0] Test Service: /gonio_orientation_solver/get_gonio_left_solution (ServiceAction)



---
## 3. Modifying Sequence Tools
Test tools for modifying existing actions in the sequence.

In [21]:
# Test 3.1: Set Action Parameters
print("=" * 60)
print("TEST 3.1: Set Action Parameters")
print("=" * 60)

# Try to set parameters on the first action (if it exists and is a service)
sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

if seq_data['total_count'] > 0:
    input_json = json.dumps({
        "index": 0,
        "parameters": {
            "test_param": "test_value",
            "numeric_param": 42
        }
    })
    
    result = tools._set_action_parameters(input_json)
    data = json.loads(result)
    
    if data.get('success'):
        print(f"✓ {data['message']}")
        print(f"  Parameters: {data['parameters']}")
    else:
        print(f"✗ Error: {data.get('error', 'Unknown error')}")
        print("  Note: This may fail if the action doesn't have these parameters")
else:
    print("✗ No actions in sequence to modify")
print()

TEST 3.1: Set Action Parameters
✗ Error: Parameter 'test_param' not found in action at index 0
  Note: This may fail if the action doesn't have these parameters



In [ ]:
# Test 3.2: Move Action
print("=" * 60)
print("TEST 3.2: Move Action")
print("=" * 60)

sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

if seq_data['total_count'] >= 2:
    print(f"Moving action from index 1 to index 0...")
    input_json = json.dumps({
        "old_index": 1,
        "new_index": 0
    })
    
    result = tools._move_action(input_json)
    data = json.loads(result)
    
    if data.get('success'):
        print(f"✓ {data['message']}")
        # Show updated sequence
        updated = tools._get_action_list("")
        updated_data = json.loads(updated)
        print("  Updated sequence:")
        for action in updated_data['actions']:
            print(f"    [{action['index']}] {action['name']}")
    else:
        print(f"✗ Error: {data.get('error', 'Unknown error')}")
else:
    print("✗ Need at least 2 actions to test move operation")
print()

In [ ]:
# Test 3.3: Delete Action
print("=" * 60)
print("TEST 3.3: Delete Action")
print("=" * 60)

sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

if seq_data['total_count'] > 0:
    # Delete the last action
    last_index = seq_data['total_count'] - 1
    print(f"Deleting action at index {last_index}...")
    
    input_json = json.dumps({"index": last_index})
    
    result = tools._delete_action(input_json)
    data = json.loads(result)
    
    if data.get('success'):
        print(f"✓ {data['message']}")
        print(f"  Remaining actions: {data['remaining_actions']}")
    else:
        print(f"✗ Error: {data.get('error', 'Unknown error')}")
else:
    print("✗ No actions to delete")
print()

---
## 4. Execution Tools
Test tools for executing the sequence. **⚠️ Use with caution on real hardware!**

In [9]:
# Test 4.1: Execute Single Action (DRY RUN - won't actually execute)
print("=" * 60)
print("TEST 4.1: Execute Single Action")
print("=" * 60)

sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

success = tools._execute_single_action(json.dumps({'index': 0}))

if seq_data['total_count'] > 0:
    print("⚠️  WARNING: This will attempt to execute an action!")
    print("⚠️  Uncomment the code below only if you want to test execution")
    print()
    print("# To test execution, uncomment this code:")
    print("# input_json = json.dumps({'index': 0})")
    print("# result = tools._execute_single_action(input_json)")
    print("# data = json.loads(result)")
    print("# if data.get('success'):")
    print("#     print(f'✓ {data['message']}')")
    print("# else:")
    print("#     print(f'✗ {data['message']}')")
else:
    print("✗ No actions to execute")
print()

TEST 4.1: Execute Single Action


[INFO] [1770651239.950189200] [tools_test_node]: Executing 'Test Service: /gonio_orientation_solver/get_gonio_left_solution'...


⚠️  WARNING: This will attempt to execute an action!
⚠️  Uncomment the code below only if you want to test execution

# To test execution, uncomment this code:
# input_json = json.dumps({'index': 0})
# result = tools._execute_single_action(input_json)
# data = json.loads(result)
# if data.get('success'):
#     print(f'✓ {data['message']}')
# else:
#     print(f'✗ {data['message']}')



[INFO] [1770651240.951508940] [tools_test_node]: Service Watchdog - node 'gonio_orientation_solver' is still active!
[INFO] [1770651240.963532197] [tools_test_node]: Test1..
[INFO] [1770651240.965558864] [tools_test_node]: Test2..
[INFO] [1770651240.966557597] [tools_test_node]: Test3..
[INFO] [1770651240.967418348] [tools_test_node]: Action Test Service: /gonio_orientation_solver/get_gonio_left_solution executed successfully!


In [ ]:
# Test 4.2: Execute Sequence (DRY RUN - won't actually execute)
print("=" * 60)
print("TEST 4.2: Execute Complete Sequence")
print("=" * 60)

sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

if seq_data['total_count'] > 0:
    print("⚠️  WARNING: This will attempt to execute the entire sequence!")
    print("⚠️  Uncomment the code below only if you want to test execution")
    print()
    print("# To test execution, uncomment this code:")
    print("# input_json = json.dumps({'start_index': 0})")
    print("# result = tools._execute_sequence(input_json)")
    print("# data = json.loads(result)")
    print("# if data.get('success'):")
    print("#     print(f'✓ Sequence executed successfully')")
    print("#     print(f'  Started at: {data['start_index']}')")
    print("#     print(f'  Finished at: {data['final_index']}')")
    print("# else:")
    print("#     print(f'✗ {data['message']}')")
else:
    print("✗ No actions to execute")
print()

---
## 5. Persistence Tools
Test tools for saving and loading sequences.

In [20]:
# Test 5.1: Save Sequence
print("=" * 60)
print("TEST 5.1: Save Sequence")
print("=" * 60)

import os
test_name = "test_rsap_sequence.json"

sequence = tools._get_action_list("")
seq_data = json.loads(sequence)

if seq_data['total_count'] > 0:
    input_json = json.dumps({"file_name": test_name})

    print(f"Saving sequence to {test_name}...")
    
    result = tools._save_sequence(input_json)
    data = json.loads(result)

    print(f"Save result: {data}")
    
    if data.get('success'):
        print(f"✓ {data['message']}")
        print(f"  Action count: {data['action_count']}")
        if os.path.exists(test_name):
            print(f"  File exists: {test_name}")
    else:
        print(f"✗ Error: {data.get('error', 'Unknown error')}")
else:
    print("✗ No actions to save")
print()

TEST 5.1: Save Sequence
Saving sequence to test_rsap_sequence.json...
Save result: {'success': False, 'error': 'file_path is required'}
✗ Error: file_path is required



In [ ]:
# Test 5.2: Clear Sequence
print("=" * 60)
print("TEST 5.2: Clear Sequence")
print("=" * 60)

result = tools._clear_sequence("")
data = json.loads(result)

if data.get('success'):
    print(f"✓ {data['message']}")
    print(f"  Remaining actions: {data['remaining_actions']}")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")
print()

In [ ]:
# Test 5.3: Load Sequence
print("=" * 60)
print("TEST 5.3: Load Sequence")
print("=" * 60)

test_file_path = "/tmp/test_rsap_sequence.json"

if os.path.exists(test_file_path):
    input_json = json.dumps({"file_path": test_file_path})
    
    result = tools._load_sequence(input_json)
    data = json.loads(result)
    
    if data.get('success'):
        print(f"✓ {data['message']}")
        print(f"  Action count: {data['action_count']}")
        
        # Show loaded sequence
        loaded = tools._get_action_list("")
        loaded_data = json.loads(loaded)
        print("  Loaded actions:")
        for action in loaded_data['actions']:
            print(f"    [{action['index']}] {action['name']}")
    else:
        print(f"✗ Error: {data.get('error', 'Unknown error')}")
else:
    print(f"✗ Test file not found: {test_file_path}")
    print("  Run Test 5.1 first to save a sequence")
print()

---
## 6. Custom Tool Tests
Use these cells to test specific tool combinations or custom scenarios.

In [ ]:
# Custom Test: Your custom tool test here
print("=" * 60)
print("CUSTOM TEST")
print("=" * 60)

# Example: Build a complete sequence programmatically
print("Building a complete test sequence...")

# 1. Clear existing sequence
tools._clear_sequence("")

# 2. Get available services
services_result = tools._get_available_services("")
services_data = json.loads(services_result)

if services_data.get('count', 0) > 0:
    # 3. Add first service
    service1 = services_data['services'][0]
    tools._add_service_to_sequence(json.dumps({
        "service_client": service1['client'],
        "index": 0,
        "service_name": "First Action"
    }))
    
    # 4. Add user interaction
    tools._add_user_interaction(json.dumps({
        "index": 1,
        "action_name": "Manual Check",
        "action_description": "Please verify the system state"
    }))
    
    # 5. Add another service if available
    if services_data['count'] > 1:
        service2 = services_data['services'][1]
        tools._add_service_to_sequence(json.dumps({
            "service_client": service2['client'],
            "index": 2,
            "service_name": "Second Action"
        }))
    
    # 6. Display final sequence
    final = tools._get_action_list("")
    final_data = json.loads(final)
    print(f"\n✓ Created sequence with {final_data['total_count']} actions:")
    for action in final_data['actions']:
        print(f"  [{action['index']}] {action['name']} ({action['type']})")
else:
    print("✗ No services available")

print()

---
## 7. Cleanup
Don't forget to cleanup ROS2 resources when done.

In [10]:
# Cleanup: Destroy node and shutdown ROS2
print("=" * 60)
print("CLEANUP")
print("=" * 60)

try:
    test_node.destroy_node()
    rclpy.shutdown()
    print("✓ ROS2 node destroyed and shutdown complete")
except Exception as e:
    print(f"✗ Cleanup error: {e}")
print("\n🎉 Testing complete!")

CLEANUP
✗ Cleanup error: Context must be initialized before it can be shutdown

🎉 Testing complete!


In [3]:
# Test 1.5: Get Parameter Value Recommendations
print("=" * 60)
print("TEST 1.5: Get Parameter Value Recommendations")
print("=" * 60)

# Get all available value recommendations
print("Getting all available parameter value recommendations...")
result = tools._get_parameter_value_recommendations_structured(parameter_type=None)
data = json.loads(result)

if data.get('success'):
    print(f"✓ Found {data['count']} value set(s)")
    for set_name, set_info in data['value_sets'].items():
        print(f"\n  Value Set: {set_name}")
        print(f"    Type: {set_info['type']}")
        print(f"    Values: {set_info['values'][:5]}")  # Show first 5 values
        if len(set_info['values']) > 5:
            print(f"    ... and {len(set_info['values']) - 5} more")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")

print("\n--- Testing type-specific recommendations ---")
# Get recommendations for string type parameters
result = tools._get_parameter_value_recommendations_structured(parameter_type="string")
data = json.loads(result)

if data.get('success'):
    print(f"✓ Found {data['count']} value set(s) compatible with type 'string'")
    for set_name in data['value_sets'].keys():
        print(f"  - {set_name}")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")

print("\n--- Testing type-specific recommendations for 'str' ---")
# Test with 'str' type (should be compatible with 'string')
result = tools._get_parameter_value_recommendations_structured(parameter_type="str")
data = json.loads(result)

if data.get('success'):
    print(f"✓ Found {data['count']} value set(s) compatible with type 'str'")
    for set_name in data['value_sets'].keys():
        print(f"  - {set_name}")
else:
    print(f"✗ Error: {data.get('error', 'Unknown error')}")

print()


TEST 1.5: Get Parameter Value Recommendations
Getting all available parameter value recommendations...
✓ Found 6 value set(s)

  Value Set: components_in_scene
    Type: str
    Values: ['POG_Glas_Dummies-1']

  Value Set: frames_in_scene
    Type: str
    Values: ['POG_Glas_Dummies-1_Center_top_helper', 'POG_Glas_Dummies-1_Gripping_Point', 'POG_Glas_Dummies-1_Laser_1', 'POG_Glas_Dummies-1_Laser_2', 'POG_Glas_Dummies-1_Laser_3']
    ... and 13 more

  Value Set: instructions_in_scene
    Type: str
    Values: []

  Value Set: tf_frames
    Type: str
    Values: ['1K_Dispenser', 'Z_Axis', '1K_Dispenser_Protection', '2K_Dispenser_Cartridge', 'Calibration_Qube']
    ... and 75 more

  Value Set: vision_cameras
    Type: str
    Values: ['pm_robot_basler_top_cam_1.yaml', 'webcam_config.yaml', 'pm_robot_basler_bottom_cam_2.yaml', 'pm_robot_top_cam_1_Unity.yaml']

  Value Set: vision_processes
    Type: str
    Values: ['test_pipeline.json', 'test_pipeline (copy).json', 'process_demo.json', 